# Quickstart: Using the Speech Service from Python

This sample shows how to use the Speech Service through the Speech SDK for Python. It illustrates how the SDK can be used to recognize speech from microphone input.

See the [accompanying article](https://docs.microsoft.com/azure/cognitive-services/speech-service/quickstart-python) on the SDK documentation page for step-by-step instructions.

## Prerequisites

Before you get started, here's a list of prerequisites:

* A subscription key for the Speech service. See [Try the speech service for free](https://docs.microsoft.com/azure/cognitive-services/speech-service/get-started).
* Python 3.5 or later needs to be installed. Downloads are available [here](https://www.python.org/downloads/).
* The Python Speech SDK package is available for Windows (x64 or x86), Mac (macOS X version 10.12 or later), and Linux (x64; Ubuntu 16.04 or Ubuntu 18.04).
* On Ubuntu 16.04 or 18.04, run the following commands for the installation of required packages:
  ```sh
  sudo apt-get update
  sudo apt-get install libssl1.0.0 libasound2
  ```
* On Debian 9, run the following commands for the installation of required packages:
  ```sh
  sudo apt-get update
  sudo apt-get install libssl1.0.2 libasound2
  ```
* On Windows you need the [Microsoft Visual C++ Redistributable for Visual Studio 2017](https://support.microsoft.com/help/2977003/the-latest-supported-visual-c-downloads) for your platform.

## Get the Speech SDK Python Package

**By downloading the Microsoft Cognitive Services Speech SDK, you acknowledge its license, see [Speech SDK license agreement](https://aka.ms/csspeech/license).**

The Cognitive Services Speech SDK Python package can be installed from [pyPI](https://pypi.org/) using this command:

```sh
pip install azure-cognitiveservices-speech
```


## Speech Recognition Using the Speech SDK

First, set up some general items. Import the Speech SDK Python:

In [21]:
import azure.cognitiveservices.speech as speechsdk
import json
import string
import time
import threading
import wave
import utils

Set up the subscription info for the Speech Service:

In [8]:
speech_key, service_region = "8s6arAOHC5EpBi8alrcJxNGFBzKlo8cu8pAk5UqFjy88QcflRSAJJQQJ99ALAC4f1cMXJ3w3AAAYACOGuHRg", "westus"

Create an instance of a speech config with specified subscription key and service region.
Replace with your own subscription key and service region (e.g., "westus").

In [9]:
speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=service_region)


In [26]:
audio_config = speechsdk.audio.AudioConfig(filename="data/sample5.wav")
speech_recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, language="en-US", audio_config=audio_config)

In [27]:
pronunciation_config = speechsdk.PronunciationAssessmentConfig( 
    reference_text="", 
    grading_system=speechsdk.PronunciationAssessmentGradingSystem.HundredMark, 
    granularity=speechsdk.PronunciationAssessmentGranularity.Phoneme, 
    enable_miscue=False) 
pronunciation_config.enable_prosody_assessment() 
# pronunciation_config.enable_content_assessment_with_topic("greeting")

In [12]:
speech_recognizer.session_started.connect(lambda evt: print(f"SESSION ID: {evt.session_id}"))
pronunciation_config.apply_to(speech_recognizer)
speech_recognition_result = speech_recognizer.recognize_once()
# The pronunciation assessment result as a Speech SDK object
pronunciation_assessment_result = speechsdk.PronunciationAssessmentResult(speech_recognition_result)

# The pronunciation assessment result as a JSON string
pronunciation_assessment_result_json = speech_recognition_result.properties.get(speechsdk.PropertyId.SpeechServiceResponse_JsonResult)

SESSION ID: 6d14420f59cb45488def4f52981f82bc


In [13]:
# create a json file from the result
with open('pronunciation_assessment_result.json', 'w') as f:
    f.write(pronunciation_assessment_result_json)

Create a recognizer with the given settings. Since no explicit audio config is specified, the default microphone will be used (make sure the audio settings are correct).

In [14]:
speech_recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config)

Info: on_underlying_io_bytes_received: Close frame received
Info: on_underlying_io_bytes_received: closing underlying io.
Info: on_underlying_io_close_complete: uws_state: 6.


In [15]:
import json
from pprint import pprint

def parse_json(file_content):
    data = json.loads(file_content)
    parsed_data = {
        "Id": data.get("Id"),
        "Recognition Status": data.get("RecognitionStatus"),
        "Text Analysis": {
            "Text": data.get("DisplayText"),
            "Duration (ms)": data.get("Duration"),
            "Confidence": data.get("NBest", [{}])[0].get("Confidence"),
        },
        "Pronunciation Assessment": {
            "Scores": data.get("NBest", [{}])[0].get("PronunciationAssessment"),
        },
        "Detailed Word Analysis": []
    }
    
    for word in data.get("NBest", [{}])[0].get("Words", []):
        word_info = {
            "Word": word.get("Word"),
            "Offset (ms)": word.get("Offset"),
            "Duration (ms)": word.get("Duration"),
            "Accuracy Score": word.get("PronunciationAssessment", {}).get("AccuracyScore"),
            "Phonemes": [
                {
                    "Phoneme": phoneme.get("Phoneme"),
                    "Accuracy Score": phoneme.get("PronunciationAssessment", {}).get("AccuracyScore"),
                    "Offset (ms)": phoneme.get("Offset"),
                    "Duration (ms)": phoneme.get("Duration"),
                }
                for phoneme in word.get("Phonemes", [])
            ],
        }
        parsed_data["Detailed Word Analysis"].append(word_info)
    
    return parsed_data

parsed_data = parse_json(pronunciation_assessment_result_json)

In [16]:
parsed_data

{'Id': '226b46d592c44fddbccd0557f3d0b51c',
 'Recognition Status': 'Success',
 'Text Analysis': {'Text': "Advertisement is very good. Maybe I translated it into me, to me and maybe I don't know, Unconsciously I want to buy the product they advertise.",
  'Duration (ms)': 145000000,
  'Confidence': 0.8275869},
 'Pronunciation Assessment': {'Scores': {'AccuracyScore': 89.0,
   'FluencyScore': 72.0,
   'ProsodyScore': 75.1,
   'CompletenessScore': 100.0,
   'PronScore': 76.0}},
 'Detailed Word Analysis': [{'Word': 'advertisement',
   'Offset (ms)': 17300000,
   'Duration (ms)': 8700000,
   'Accuracy Score': 76.0,
   'Phonemes': [{'Phoneme': 'ax',
     'Accuracy Score': 66.0,
     'Offset (ms)': 17300000,
     'Duration (ms)': 1300000},
    {'Phoneme': 'd',
     'Accuracy Score': 100.0,
     'Offset (ms)': 18700000,
     'Duration (ms)': 400000},
    {'Phoneme': 'v',
     'Accuracy Score': 100.0,
     'Offset (ms)': 19200000,
     'Duration (ms)': 600000},
    {'Phoneme': 'er',
     'Accura

In [28]:
#save parsed data into a json file
with open('parsed_pronunciation_assessment_result.json', 'w') as f:
    json.dump(parsed_data, f, indent=4)

In [29]:
parsed_data["Text Analysis"]["Text"]

"Advertisement is very good. Maybe I translated it into me, to me and maybe I don't know, Unconsciously I want to buy the product they advertise."

In [64]:
from openai import OpenAI

client = OpenAI(
  base_url = "https://integrate.api.nvidia.com/v1",
  api_key = "k-proj-W3YsGTDDHw4YmjCAv0dRx7QtVQgaQ_3p-CjbtF5Z1_VhkitgH5-QzGdyIHtJpsi_JuK71w9zw3T3BlbkFJ-gd-r6ugTs8I34RTDwJIVA0PNKtKtoBri1PgOaFaAkcTZvxxHdtPDmiMpQuWgxcEAc9q9u2JsA"
)

completion = client.chat.completions.create(
  model="nvdev/meta/llama-3.1-70b-instruct",
  messages=[{"role":"user","content":"Write a limerick about the wonders of GPU computing."}],
  temperature=0.2,
  top_p=0.7,
  max_tokens=1024,
  stream=True
)

for chunk in completion:
  if chunk.choices[0].delta.content is not None:
    print(chunk.choices[0].delta.content, end="")

AuthenticationError: Error code: 401 - {'status': 401, 'title': 'Unauthorized', 'detail': 'Invalid JWT serialization: Missing dot delimiter(s)'}

In [74]:
criteria = '''
Score
Fluency and coherence Lexical resource Grammatical range and
accuracy
Pronunciation
9
Fluent with only very occasional repetition or
self-correction.
Any hesitation that occurs is used only to
prepare the content of the next utterance and
not to find words or grammar.
Speech is situationally appropriate and
cohesive features are fully acceptable.
Topic development is fully coherent and
appropriately extended.
Total flexibility and precise use in all contexts.
Sustained use of accurate and idiomatic
language.
Structures are precise and accurate at all times,
apart from mistakes characteristic of native
speaker speech.
Uses a full range of phonological features to
convey precise and/or subtle meaning.
Flexible use of features of connected speech is
sustained throughout.
Can be effortlessly understood throughout.
Accent has no effect on intelligibility.
8
Fluent with only very occasional repetition or
self-correction.
Hesitation may occasionally be used to find
words or grammar, but most will be content
related.
Topic development is coherent, appropriate
and relevant.
Wide resource, readily and flexibly used to
discuss all topics and convey precise meaning.
Skilful use of less common and idiomatic items
despite occasional inaccuracies in word choice
and collocation.
Effective use of paraphrase as required.
Wide range of structures, flexibly used.
The majority of sentences are error free.
Occasional inappropriacies and non-systematic
errors occur. A few basic errors may persist.
Uses a wide range of phonological features to
convey precise and/or subtle meaning.
Can sustain appropriate rhythm. Flexible use of
stress and intonation across long utterances,
despite occasional lapses.
Can be easily understood throughout.
Accent has minimal effect on intelligibility.
7
Able to keep going and readily produce long
turns without noticeable effort.
Some hesitation, repetition and/or selfcorrection may occur, often mid-sentence and
indicate problems with accessing appropriate
language. However, these will not affect
coherence.
Flexible use of spoken discourse markers,
connectives and cohesive features.
Resource flexibly used to discuss a variety of
topics.
Some ability to use less common and idiomatic
items and an awareness of style and collocation
is evident though inappropriacies occur.
Effective use of paraphrase as required.
A range of structures flexibly used. Error-free
sentences are frequent.
Both simple and complex sentences are used
effectively despite some errors. A few basic
errors persist.
Displays all the positive features of band 6, and
some, but not all, of the positive features of
band 8.
d
Score
Fluency and coherence Lexical resource Grammatical range and
accuracy
Pronunciation
6
Able to keep going and demonstrates a willingness to
produce long turns.
Coherence may be lost at times as a result of hesitation,
repetition and/or self-correction.
Uses a range of spoken discourse markers, connectives
and cohesive features though not always appropriately.
Resource sufficient to discuss topics at length.
Vocabulary use may be inappropriate but meaning is
clear.
Generally able to paraphrase successfully.
Produces a mix of short and complex sentence forms
and a variety of structures with limited flexibility.
Though errors frequently occur in complex structures,
these rarely impede communication.
Uses a range of phonological features, but control is
variable.
Chunking is generally appropriate, but rhythm may be
affected by a lack of stress-timing and/or a rapid speech
rate.
Some effective use of intonation and stress, but this is
not sustained.
Individual words or phonemes may be mispronounced
but this causes only occasional lack of clarity.
Can generally be understood throughout without much
effort.
5
Usually able to keep going, but relies on repetition and
self-correction to do so and/or on slow speech.
Hesitations are often associated with mid-sentence
searches for fairly basic lexis and grammar.
Overuse of certain discourse markers, connectives and
other cohesive features.
More complex speech usually causes disfluency but
simpler language may be produced fluently.
Resource sufficient to discuss familiar and unfamiliar
topics but there is limited flexibility.
Attempts paraphrase but not always with success.
Basic sentence forms are fairly well controlled for
accuracy.
Complex structures are attempted but these are limited
in range, nearly always contain errors and may lead to
the need for reformulation.
Displays all the positive features of band 4, and some,
but not all, of the positive features of band 6.
4
Unable to keep going without noticeable pauses.
Speech may be slow with frequent repetition.
Often self-corrects.
Can link simple sentences but often with repetitious use
of connectives.
Some breakdowns in coherence.
Resource sufficient for familiar topics but only basic
meaning can be conveyed on unfamiliar topics.
Frequent inappropriacies and errors in word choice.
Rarely attempts paraphrase.
Can produce basic sentence forms and some short
utterances are error-free.
Subordinate clauses are rare and, overall, turns are
short, structures are repetitive and errors are frequent.
Uses some acceptable phonological features, but the
range is limited.
Produces some acceptable chunking, but there are
frequent lapses in overall rhythm.
Attempts to use intonation and stress, but control is
limited.
Individual words or phonemes are frequently
mispronounced, causing lack of clarity.
Understanding requires some effort and there may be
patches of speech that cannot be understood.
3
Frequent, sometimes long, pauses occur while
candidate searches for words.
Limited ability to link simple sentences and go
beyond simple responses to questions.
Frequently unable to convey basic message.
Resource limited to simple vocabulary used
primarily to convey personal information.
Vocabulary inadequate for unfamiliar topics.
Basic sentence forms are attempted but
grammatical errors are numerous except in
apparently memorised utterances.
Displays some features of band 2, and some,
but not all, of the positive features of band 4.
2
Lengthy pauses before nearly every word.
Isolated words may be recognisable but speech
is of virtually no communicative significance.
Very limited resource. Utterances consist of
isolated words or memorised utterances.
Little communication possible without the
support of mime or gesture.
No evidence of basic sentence forms. Uses few acceptable phonological features
(possibly because sample is insufficient).
Overall problems with delivery impair attempts
at connected speech.
Individual words and phonemes are mainly
mispronounced and little meaning is conveyed.
Often unintelligible.
1
Essentially none.
Speech is totally incoherent.
No resource bar a few isolated words.
No communication possible.
No rateable language unless memorised. Can produce occasional individual words and
phonemes that are recognisable, but no overall
meaning is conveyed.
Unintelligible.
'''

In [80]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are a knowledgeable, helpful assistant to help non-native English speakers to improve their English-speaking skills"
    "Use the criteria to grade the pronunciation of the user's speech. Provide feedback on the user's pronunciation, fluency, coherence, lexical resource, grammatical range and accuracy."
    "{criteria}"
    # "the question. If you don't know the answer, say that you "
    # "don't know. Use three sentences maximum and keep the "
    # "answer concise."
    # "\n\n"
    # "{context}"
        )

human_prompt = (
    "Use the following data extracted from a speaking session to provide feedback on the user's pronunciation, fluency, coherence, lexical resource, grammatical range and accuracy. Point out where the users can lose points according to the critieria and how they can improve."
    "{parsed_data}"
    # "the question. If you don't know the answer, say that you "
    # "don't know. Use three sentences maximum and keep the "
    # "answer concise."
    # "\n\n"
    # "{context}"
        )
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", human_prompt),
    ]
)
client = ChatNVIDIA(
  model="nvdev/meta/llama-3.1-405b-instruct",
  api_key="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", 
  temperature=0.2,
  top_p=0.7,
  max_tokens=1024,
)

input_data = {
    "criteria": criteria,
    "parsed_data": parsed_data
}
formatted_prompt = prompt.format(criteria=criteria, parsed_data=parsed_data)
print(type(formatted_prompt))
client.invoke(formatted_prompt)

# for chunk in client.stream([{"role":"user","content":"Write a limerick about the wonders of GPU computing."}]): 
#   print(chunk.content, end="")

<class 'str'>


AIMessage(content='Based on the provided data, I\'ll give you feedback on your pronunciation, fluency, coherence, lexical resource, grammatical range, and accuracy.\n\n**Pronunciation: 7**\nYour pronunciation is generally good, but there are some areas that need improvement. You scored 76.0 in PronScore, which indicates that you have some difficulties with phonemes, especially with the sounds /ax/, /r/, and /v/. For example, in the word "advertisement", your accuracy score for the phoneme /ax/ was 50.0, and for /r/ was 41.0. To improve, focus on practicing these sounds in isolation and in words.\n\n**Fluency: 7**\nYour fluency is good, but you sometimes hesitate or repeat words, which affects your overall fluency. Your FluencyScore was 72.0, which indicates that you need to work on speaking more smoothly and naturally. Practice speaking at a moderate pace and try to avoid fillers (e.g., "um", "ah").\n\n**Coherence: 8**\nYour coherence is good, and you were able to convey your ideas cle

In [66]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

client = ChatNVIDIA(
  model="nvdev/meta/llama-3.1-70b-instruct",
  api_key="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", 
  temperature=0.2,
  top_p=0.7,
  max_tokens=1024,
)

client.invoke(["Write me a poem"])

# for chunk in client.stream([{"role":"user","content":"Write a limerick about the wonders of GPU computing."}]): 
#   print(chunk.content, end="")


AIMessage(content='Here is a poem I just came up with:\n\n"Moonlit Dreams"\n\nThe night is dark, the stars are bright\nA silver glow, a gentle light\nThe moon, a crescent in the sky\nLulls me to sleep, with a gentle sigh\n\nMy dreams are filled, with visions sweet\nOf distant lands, and memories to keep\nThe world is hushed, in quiet sleep\nAs I drift off, my soul to keep\n\nThe wind whispers secrets, in my ear\nOf far-off places, and memories so dear\nThe moon\'s soft light, shines down on me\n Illuminating, all that\'s meant to be\n\nIn this peaceful night, I find my rest\nMy heart and soul, are at their best\nThe world may be, a busy place\nBut in the stillness, I find my space\n\nSo let the moon, shine down on me\nAnd fill my dreams, with serenity\nFor in its light, I am free to roam\nAnd find my peace, in this quiet home.\n\nI hope you enjoy it!', additional_kwargs={}, response_metadata={'role': 'assistant', 'content': 'Here is a poem I just came up with:\n\n"Moonlit Dreams"\n\nTh

In [56]:
import os

os.environ["OPENAI_API_VERSION"] = "2023-12-01-preview"
os.environ["AZURE_OPENAI_ENDPOINT"] = "http://localhost:8000/v1"
os.environ["AZURE_OPENAI_API_KEY"] = "k-proj-W3YsGTDDHw4YmjCAv0dRx7QtVQgaQ_3p-CjbtF5Z1_VhkitgH5-QzGdyIHtJpsi_JuK71w9zw3T3BlbkFJ-gd-r6ugTs8I34RTDwJIVA0PNKtKtoBri1PgOaFaAkcTZvxxHdtPDmiMpQuWgxcEAc9q9u2JsA"

In [57]:
#export openai key


AttributeError: module 'os' has no attribute 'export'

In [55]:
# Import Azure OpenAI
from langchain_openai import AzureOpenAI
llm = AzureOpenAI(
    deployment_name="gpt-35-turbo-instruct-0914",
)

# Run the LLM
llm.invoke("Tell me a joke")

APIConnectionError: Connection error.

In [48]:
import openai

client = openai.AzureOpenAI(
    api_version="2023-12-01-preview",
)

response = client.completions.create(
    model="gpt-35-turbo-instruct-prod",
    prompt="Test prompt"
)

APIConnectionError: Connection error.

In [ ]:
import openai

client = openai.AzureOpenAI(
    api_version="2023-12-01-preview",
)

response = client.completions.create(
    model="gpt-35-turbo-instruct-prod",
    prompt="Test prompt"
)

Starts speech recognition, and returns after a single utterance is recognized. The end of a
single utterance is determined by listening for silence at the end or until a maximum of about 30
seconds of audio is processed.  The task returns the recognition text as result. 
Note: Since `recognize_once()` returns only a single utterance, it is suitable only for single
shot recognition like command or query. 
For long-running multi-utterance recognition, use `start_continuous_recognition()` instead.

In [17]:
result = speech_recognizer.recognize_once()

KeyboardInterrupt: 

In [ ]:
if result.reason == speechsdk.ResultReason.RecognizedSpeech:
    print("Recognized: {}".format(result.text))
elif result.reason == speechsdk.ResultReason.NoMatch:
    print("No speech could be recognized: {}".format(result.no_match_details))
elif result.reason == speechsdk.ResultReason.Canceled:
    cancellation_details = result.cancellation_details
    print("Speech Recognition canceled: {}".format(cancellation_details.reason))
    if cancellation_details.reason == speechsdk.CancellationReason.Error:
        print("Error details: {}".format(cancellation_details.error_details))

No speech could be recognized: NoMatchDetails(reason=NoMatchReason.NotRecognized)


Info: on_underlying_io_bytes_received: Close frame received
Info: on_underlying_io_bytes_received: received close frame, sending a close response frame.
Info: on_underlying_io_close_sent: uws_client=0x106c67e20, io_send_result:0
Info: on_underlying_io_close_sent: closing underlying io.
Info: on_underlying_io_close_complete: uws_state: 6.


In [ ]:
def pronunciation_assessment_from_microphone():
    """Performs one-shot pronunciation assessment asynchronously with input from microphone.
        See more information at https://aka.ms/csspeech/pa"""

    # Creates an instance of a speech config with specified subscription key and service region.
    # Replace with your own subscription key and service region (e.g., "westus").
    config = speechsdk.SpeechConfig(subscription=speech_key, region=service_region)

    # The pronunciation assessment service has a longer default end silence timeout (5 seconds) than normal STT
    # as the pronunciation assessment is widely used in education scenario where kids have longer break in reading.
    # You can adjust the end silence timeout based on your real scenario.
    config.set_property(speechsdk.PropertyId.SpeechServiceConnection_EndSilenceTimeoutMs, "3000")

    reference_text = ""
    pronunciation_config = speechsdk.PronunciationAssessmentConfig(
        reference_text=reference_text,
        grading_system=speechsdk.PronunciationAssessmentGradingSystem.HundredMark,
        granularity=speechsdk.PronunciationAssessmentGranularity.Phoneme,
        enable_miscue=True)
    pronunciation_config.enable_prosody_assessment()

    # Create a speech recognizer, also specify the speech language
    recognizer = speechsdk.SpeechRecognizer(speech_config=config, language="en-US")
    while True:
        # Receives reference text from console input.
        print('Enter reference text you want to assess, or enter empty text to exit.')
        print('> ', end='')

        try:
            reference_text = input()
        except EOFError:
            break

        if not reference_text:
            break

        pronunciation_config.reference_text = reference_text
        # (Optional) get the session ID
        recognizer.session_started.connect(lambda evt: print(f"SESSION ID: {evt.session_id}"))
        pronunciation_config.apply_to(recognizer)

        # Starts recognizing.
        print('Read out "{}" for pronunciation assessment ...'.format(reference_text))

        # Note: Since recognize_once() returns only a single utterance, it is suitable only for single
        # shot evaluation.
        # For long-running multi-utterance pronunciation evaluation, use start_continuous_recognition() instead.
        result = recognizer.recognize_once_async().get()

        # Check the result
        if result.reason == speechsdk.ResultReason.RecognizedSpeech:
            print('Recognized: {}'.format(result.text))
            print('  Pronunciation Assessment Result:')

            pronunciation_result = speechsdk.PronunciationAssessmentResult(result)
            print('    Accuracy score: {}, Prosody score: {}, Pronunciation score: {}, Completeness score : {}, FluencyScore: {}'.format(
                pronunciation_result.accuracy_score, pronunciation_result.prosody_score, pronunciation_result.pronunciation_score,
                pronunciation_result.completeness_score, pronunciation_result.fluency_score
            ))
            print('  Word-level details:')
            for idx, word in enumerate(pronunciation_result.words):
                print('    {}: word: {}, accuracy score: {}, error type: {};'.format(
                    idx + 1, word.word, word.accuracy_score, word.error_type
                ))
        elif result.reason == speechsdk.ResultReason.NoMatch:
            print("No speech could be recognized")
        elif result.reason == speechsdk.ResultReason.Canceled:
            cancellation_details = result.cancellation_details
            print("Speech Recognition canceled: {}".format(cancellation_details.reason))
            if cancellation_details.reason == speechsdk.CancellationReason.Error:
                print("Error details: {}".format(cancellation_details.error_details))

In [ ]:
def pronunciation_assessment_continuous_from_file(filename):
    """Performs continuous pronunciation assessment asynchronously with input from an audio file.
        See more information at https://aka.ms/csspeech/pa"""

    import difflib
    import json

    # Creates an instance of a speech config with specified subscription key and service region.
    # Replace with your own subscription key and service region (e.g., "westus").
    # Note: The sample is for en-US language.
    speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=service_region)
    audio_config = speechsdk.audio.AudioConfig(filename=filename)

    with open(filename, "r", encoding="utf-8") as t:
        reference_text = t.readline()
    # Create pronunciation assessment config, set grading system, granularity and if enable miscue based on your requirement.
    enable_miscue = True
    enable_prosody_assessment = True
    pronunciation_config = speechsdk.PronunciationAssessmentConfig(
        reference_text=reference_text,
        grading_system=speechsdk.PronunciationAssessmentGradingSystem.HundredMark,
        granularity=speechsdk.PronunciationAssessmentGranularity.Phoneme,
        enable_miscue=enable_miscue)
    if enable_prosody_assessment:
        pronunciation_config.enable_prosody_assessment()

    # Creates a speech recognizer using a file as audio input.
    language = 'zh-CN'
    speech_recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, language=language, audio_config=audio_config)
    # Apply pronunciation assessment config to speech recognizer
    pronunciation_config.apply_to(speech_recognizer)

    done = False
    recognized_words = []
    prosody_scores = []
    fluency_scores = []
    durations = []
    startOffset = 0
    endOffset = 0

    def stop_cb(evt: speechsdk.SessionEventArgs):
        """callback that signals to stop continuous recognition upon receiving an event `evt`"""
        print('CLOSING on {}'.format(evt))
        nonlocal done
        done = True

    def recognized(evt: speechsdk.SpeechRecognitionEventArgs):
        print("pronunciation assessment for: {}".format(evt.result.text))
        pronunciation_result = speechsdk.PronunciationAssessmentResult(evt.result)
        print("    Accuracy score: {}, prosody score: {}, pronunciation score: {}, completeness score : {}, fluency score: {}".format(
            pronunciation_result.accuracy_score, pronunciation_result.prosody_score, pronunciation_result.pronunciation_score,
            pronunciation_result.completeness_score, pronunciation_result.fluency_score
        ))
        nonlocal recognized_words, prosody_scores, fluency_scores, durations, startOffset, endOffset
        recognized_words += pronunciation_result.words
        fluency_scores.append(pronunciation_result.fluency_score)
        if pronunciation_result.prosody_score is not None:
            prosody_scores.append(pronunciation_result.prosody_score)
        json_result = evt.result.properties.get(speechsdk.PropertyId.SpeechServiceResponse_JsonResult)
        jo = json.loads(json_result)
        nb = jo["NBest"][0]
        durations.extend([int(w["Duration"]) + 100000 for w in nb["Words"]])
        if startOffset == 0:
            startOffset = nb["Words"][0]["Offset"]
        endOffset = nb["Words"][-1]["Offset"] + nb["Words"][-1]["Duration"] + 100000

    # Connect callbacks to the events fired by the speech recognizer
    speech_recognizer.recognized.connect(recognized)
    # (Optional) get the session ID
    speech_recognizer.session_started.connect(lambda evt: print(f"SESSION ID: {evt.session_id}"))
    speech_recognizer.session_stopped.connect(lambda evt: print('SESSION STOPPED {}'.format(evt)))
    speech_recognizer.canceled.connect(lambda evt: print('CANCELED {}'.format(evt)))
    # Stop continuous recognition on either session stopped or canceled events
    speech_recognizer.session_stopped.connect(stop_cb)
    speech_recognizer.canceled.connect(stop_cb)

    # Start continuous pronunciation assessment
    speech_recognizer.start_continuous_recognition()
    while not done:
        time.sleep(.5)

    speech_recognizer.stop_continuous_recognition()

    # We need to convert the reference text to lower case, and split to words, then remove the punctuations.
    if language == 'zh-CN':
        # Split words for Chinese using the reference text and any short wave file
        reference_words = get_reference_words(zhcnfilename, reference_text, language)
    else:
        reference_words = [w.strip(string.punctuation) for w in reference_text.lower().split()]

    # For continuous pronunciation assessment mode, the service won't return the words with `Insertion` or `Omission`
    # even if miscue is enabled.
    # We need to compare with the reference text after received all recognized words to get these error words.
    if enable_miscue:
        diff = difflib.SequenceMatcher(None, reference_words, [x.word.lower() for x in recognized_words])
        final_words = []
        for tag, i1, i2, j1, j2 in diff.get_opcodes():
            if tag in ['insert', 'replace']:
                for word in recognized_words[j1:j2]:
                    word._error_type = 'Insertion'
                    final_words.append(word)
            if tag in ['delete', 'replace']:
                for word_text in reference_words[i1:i2]:
                    word = speechsdk.PronunciationAssessmentWordResult({
                        'Word': word_text,
                        'PronunciationAssessment': {
                            'ErrorType': 'Omission',
                        }
                    })
                    final_words.append(word)
            if tag == 'equal':
                final_words += recognized_words[j1:j2]
    else:
        final_words = recognized_words

    durations_sum = sum([d for w, d in zip(recognized_words, durations) if w.error_type == "None"])

    # We can calculate whole accuracy by averaging
    final_accuracy_scores = []
    for word in final_words:
        if word.error_type == 'Insertion':
            continue
        else:
            final_accuracy_scores.append(word.accuracy_score)
    accuracy_score = sum(final_accuracy_scores) / len(final_accuracy_scores)
    # Re-calculate the prosody score by averaging
    if len(prosody_scores) == 0:
        prosody_score = float("nan")
    else:
        prosody_score = sum(prosody_scores) / len(prosody_scores)
    # Re-calculate fluency score
    fluency_score = 0
    if startOffset > 0:
        fluency_score = durations_sum / (endOffset - startOffset) * 100
    # Calculate whole completeness score
    handled_final_words = [w.word for w in final_words if w.error_type != "Insertion"]
    completeness_score = len([w for w in final_words if w.error_type == "None"]) / len(handled_final_words) * 100
    completeness_score = completeness_score if completeness_score <= 100 else 100
    sorted_scores = sorted([accuracy_score, prosody_score, completeness_score, fluency_score])
    pronunciation_score = sorted_scores[0] * 0.4 + sorted_scores[1] * 0.2 + sorted_scores[2] * 0.2 + sorted_scores[3] * 0.2

    print('    Paragraph pronunciation score: {:.2f}, accuracy score: {:.2f}, prosody score: {:.2f}, completeness score: {:.2f}, fluency score: {:.2f}'.format(
        pronunciation_score, accuracy_score, prosody_score, completeness_score, fluency_score
    ))

    for idx, word in enumerate(final_words):
        print('    {}: word: {}\taccuracy score: {}\terror type: {};'.format(
            idx + 1, word.word, word.accuracy_score, word.error_type
        ))



In [24]:
def push_stream_writer(stream, filename):
    # The number of bytes to push per buffer
    n_bytes = 3200
    wav_fh = wave.open(filename)
    # Start pushing data until all data has been read from the file
    try:
        while True:
            frames = wav_fh.readframes(n_bytes // 2)
            print('read {} bytes'.format(len(frames)))
            if not frames:
                break
            stream.write(frames)
            time.sleep(.1)
    finally:
        wav_fh.close()
        stream.close()  # must be done to signal the end of stream

def read_wave_header(file_path):
    with wave.open(file_path, 'rb') as audio_file:
        framerate = audio_file.getframerate()
        bits_per_sample = audio_file.getsampwidth() * 8
        num_channels = audio_file.getnchannels()
        return framerate, bits_per_sample, num_channels

def pronunciation_assessment_from_stream(wavefilename):
    """Performs pronunciation assessment asynchronously with input from an audio stream.
        See more information at https://aka.ms/csspeech/pa"""

    # Creates an instance of a speech config with specified subscription key and service region.
    # Replace with your own subscription key and service region (e.g., "westus").
    # Note: The sample is for en-US language.
    speech_config = speechsdk.SpeechConfig(subscription=speech_key, region=service_region)

    # Setup the audio stream
    framerate, bits_per_sample, num_channels = read_wave_header(wavefilename)
    format = speechsdk.audio.AudioStreamFormat(samples_per_second=framerate, bits_per_sample=bits_per_sample, channels=num_channels)
    stream = speechsdk.audio.PushAudioInputStream(format)
    audio_config = speechsdk.audio.AudioConfig(stream=stream)

    reference_text = "What's the weather like?"
    # Create pronunciation assessment config, set grading system, granularity and if enable miscue based on your requirement.
    pronunciation_config = speechsdk.PronunciationAssessmentConfig(
        reference_text=reference_text,
        grading_system=speechsdk.PronunciationAssessmentGradingSystem.HundredMark,
        granularity=speechsdk.PronunciationAssessmentGranularity.Phoneme,
        enable_miscue=True)
    pronunciation_config.enable_prosody_assessment()

    # Create a speech recognizer using a file as audio input.
    language = 'en-US'
    speech_recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, language=language, audio_config=audio_config)
    # (Optional) get the session ID
    speech_recognizer.session_started.connect(lambda evt: print(f"SESSION ID: {evt.session_id}"))
    # Apply pronunciation assessment config to speech recognizer
    pronunciation_config.apply_to(speech_recognizer)

    # Start push stream writer thread
    push_stream_writer_thread = threading.Thread(target=push_stream_writer, args=[stream, wavefilename])
    push_stream_writer_thread.start()
    result = speech_recognizer.recognize_once_async().get()
    push_stream_writer_thread.join()

    # Check the result
    if result.reason == speechsdk.ResultReason.RecognizedSpeech:
        print('pronunciation assessment for: {}'.format(result.text))
        pronunciation_result = speechsdk.PronunciationAssessmentResult(result)
        print('    Accuracy score: {}, prosody score: {}, pronunciation score: {}, completeness score : {}, fluency score: {}'.format(
            pronunciation_result.accuracy_score, pronunciation_result.prosody_score, pronunciation_result.pronunciation_score,
            pronunciation_result.completeness_score, pronunciation_result.fluency_score
        ))
        print('  Word-level details:')
        for idx, word in enumerate(pronunciation_result.words):
            print('    {}: word: {}\taccuracy score: {}\terror type: {};'.format(
                idx + 1, word.word, word.accuracy_score, word.error_type
            ))
    elif result.reason == speechsdk.ResultReason.NoMatch:
        print("No speech could be recognized")
    elif result.reason == speechsdk.ResultReason.Canceled:
        cancellation_details = result.cancellation_details
        print("Speech Recognition canceled: {}".format(cancellation_details.reason))
        if cancellation_details.reason == speechsdk.CancellationReason.Error:
            print("Error details: {}".format(cancellation_details.error_details))

In [25]:
pronunciation_assessment_from_stream("/Users/duynguyen/ai-for-non-native-english-speaking-feedback/data/sample5.wav")

read 3200 bytes
SESSION ID: 725c97e8bda048be923232b03a0827ea
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
Info: on_underlying_io_bytes_received: Close frame received
Info: on_underlying_io_bytes_received: received close frame, sending a close response frame.
Info: on_underlying_io_close_sent: uws_client=0x106110d20, io_send_result:0
Info: on_underlying_io_close_sent: closing underlying io.
Info: on_underlying_io_close_complete: uws_state: 6.
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 3200 bytes
read 32

In [ ]:
pronunciation_assessment_from_microphone()

Enter reference text you want to assess, or enter empty text to exit.
> Read out "n/a" for pronunciation assessment ...
SESSION ID: d28f874c7c84485fa448eb587169382e
Recognized: N/A.
  Pronunciation Assessment Result:
    Accuracy score: 62.0, Prosody score: 59.3, Pronunciation score: 76.1, Completeness score : 100.0, FluencyScore: 100.0
  Word-level details:
    1: word: n/a, accuracy score: 62.0, error type: None;
Enter reference text you want to assess, or enter empty text to exit.
> 

In [19]:
pronunciation_assessment_continuous_from_file("/Users/duynguyen/ai-for-non-native-english-speaking-feedback/data/sample5.wav")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xde in position 4: invalid continuation byte